# Liver CT Segmentation & Classification
---

## 1. Dataset Preparation
---

### 1.1 Import libraries and dataset paths



In [ ]:
# 1. 导入本笔记本要用到的工具
from pathlib import Path
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt

# 2. 指定原始数据和输出图片的文件夹
source_dir = Path("raw-nii")
output_dir = Path("yh-dataset")

# 3. 确认原始数据存在，并准备输出文件夹
if not source_dir.is_dir():
    raise FileNotFoundError("Run this notebook from the project folder containing raw-nii")
output_dir.mkdir(exist_ok=True)


### 1.2 Read NIfTI volumes


In [ ]:
import nibabel as nib

def open_nii(path):
    # 1. 打开 NIfTI 文件；这一步还不会把整个 3D 影像读进内存。
    nii_img = nib.load(path)

    # 2. 读取尚未乘 slope、加 intercept 的原始像素。
    # NiBabel 的轴顺序是 (X, Y, Z)；转成 (Z, Y, X)，
    # 这样 1.3 的 shape[0] 和 1.4 的 volume[z] 都代表 Z 轴切片。
    raw_pixels = np.asarray(nii_img.dataobj.get_unscaled())
    pixels = raw_pixels.transpose(2, 1, 0)

    # 3. 从 dataobj 读取实际缩放值，留给 1.4 转换成 HU。
    # NiBabel 载入文件后，header 中这两个值通常已变成 NaN。
    slope = float(nii_img.dataobj.slope)
    intercept = float(nii_img.dataobj.inter)

    return pixels, slope, intercept


### 1.3 Select and validate paired slice indices


In [ ]:
# 1. 定义每个患者需要抽取的 6 个切片层号 (Z 轴索引)
z_slices = {
    0: [56, 58, 60, 63, 66, 69],
    1: [58, 60, 62, 64, 66, 69],
    2: [389, 404, 419, 434, 446, 452],
    3: [361, 399, 410, 425, 440, 452],
    4: [450, 470, 490, 510, 535, 550],
    5: [394, 409, 424, 439, 454, 458],
    6: [382, 396, 410, 430, 450, 470],
}

# 2. 逐一验证每位患者的数据完整性
for patient_id, slice_indices in z_slices.items():
    # 构造 CT 影像与分割标签 (mask) 的文件路径[cite: 2]
    ct_path = source_dir / f"volume-{patient_id}.nii"
    mask_path = source_dir / f"segmentation-{patient_id}.nii"
    
    # 读取影像（忽略 CT 缩放系数 slope 与 intercept，只取 3D 像素数组）
    ct_volume, slope, intercept = open_nii(ct_path)
    mask_volume, slope, intercept = open_nii(mask_path)
    
    # 规则 1：检查 CT 影像和分割标签的尺寸 (维度) 是否完全一致
    if ct_volume.shape != mask_volume.shape:
        raise ValueError(f"CT and mask shapes differ for patient {patient_id}")
    
    # 规则 2：检查是否刚好指定了 6 个切片
    if len(slice_indices) != 6:
        raise ValueError(f"Patient {patient_id} needs six z indices")
    
    # 规则 3：检查这 6 个切片层号是否有重复项 (set 集合会自动去重)
    if len(set(slice_indices)) < 6:
        raise ValueError(f"Patient {patient_id} needs six unique z indices")
    
    # 规则 4：检查切片层号是否超出了 Z 轴总层数范围
    total_slices = ct_volume.shape[0]
    for z in slice_indices:
        if z < 0 or z >= total_slices:
            raise ValueError(f"A z index is outside patient {patient_id}'s volume")

print("All seven patient lists are valid.")

### 1.4 Extract CT slices and ground-truth masks


In [ ]:
# 1. 准备保存 CT 切片的字典；键是（患者编号，切片序号）
# CT 数值先保留为 HU，下一节才转换成 0–255 的灰度图。
ct_slices = {}

# 2. 设置标签图片的颜色：0=黑色背景，1=绿色肝脏，2=红色肿瘤
mask_palette = [0, 0, 0, 0, 200, 0, 230, 40, 40] + [0] * (253 * 3)

# 3. 按 1.3 节指定的 Z 轴层号，抽取每位患者的 CT 和标签
for patient_id, z_list in z_slices.items():
    ct_path = source_dir / f"volume-{patient_id}.nii"
    mask_path = source_dir / f"segmentation-{patient_id}.nii"
    volume, slope, intercept = open_nii(ct_path)
    segmentation, _, _ = open_nii(mask_path)

    for slice_number, z in enumerate(z_list, start=1):
        # CT 和标签必须使用同一个 Z 轴层号。
        ct_hu = np.asarray(volume[z], dtype=np.float32) * slope + intercept
        mask = np.asarray(segmentation[z], dtype=np.uint8)
        ct_slices[(patient_id, slice_number)] = ct_hu

        # 保留标签值 0、1、2，并给 PNG 设置颜色表。
        mask_name = f"segmentation-{patient_id}-slice-{slice_number}.png"
        mask_image = Image.frombytes("P", (mask.shape[1], mask.shape[0]), mask.tobytes())
        mask_image.putpalette(mask_palette)
        mask_image.save(output_dir / mask_name)

    print(f"Patient {patient_id}: extracted {len(z_list)} paired slices")


## 2. Phase 1: CT Preprocessing and Enhancement
---

### 2.1 HU Windowing & Grayscale Normalization


In [ ]:
# 1. 所有 CT 切片使用同一个软组织 HU 窗口
window_min_hu = -150
window_max_hu = 250

# 2. 把低于或高于窗口范围的值限制在边界上
def apply_hu_window(ct_hu):
    windowed_ct = np.clip(ct_hu, window_min_hu, window_max_hu)
    return windowed_ct


In [ ]:
# 3. 把 HU 窗口映射成 0–255，便于保存成 8 位灰度 PNG
def normalize_for_png(windowed_ct):
    window_width = window_max_hu - window_min_hu
    scaled = (windowed_ct - window_min_hu) / window_width
    grayscale_image = (scaled * 255).astype(np.uint8)
    return grayscale_image


In [ ]:
# 4. 逐张保存归一化后的 CT 图片
for (patient_id, slice_number), ct_hu in ct_slices.items():
    image_name = f"volume-{patient_id}-slice-{slice_number}.png"
    windowed_ct = apply_hu_window(ct_hu)
    normalized_ct = normalize_for_png(windowed_ct)
    Image.fromarray(normalized_ct).save(output_dir / image_name)

print(f"Saved {len(ct_slices)} CT PNGs and matching segmentation PNGs")


### 2.2 Spatial Filtering & Contrast Enhancement


In [ ]:
# PSNR：比较滤波前后的图片。数值越高，代表两张图片越相似。
def calculate_psnr(reference_image, filtered_image):
    if reference_image.shape != filtered_image.shape:
        raise ValueError("PSNR images must have the same shape")
    return cv2.PSNR(reference_image, filtered_image)

# CNR：比较肝脏与周围区域的平均亮度差，并考虑两个区域的噪声。
def calculate_cnr(image, liver_region, surrounding_region):
    if image.shape != liver_region.shape or image.shape != surrounding_region.shape:
        raise ValueError("Image and CNR regions must have the same shape")

    liver_pixels = image[liver_region].astype(np.float64)
    surrounding_pixels = image[surrounding_region].astype(np.float64)
    if liver_pixels.size == 0 or surrounding_pixels.size == 0:
        raise ValueError("Both CNR regions must contain pixels")

    contrast = abs(liver_pixels.mean() - surrounding_pixels.mean())
    noise = np.sqrt(liver_pixels.var() + surrounding_pixels.var())
    if noise > 0:
        return contrast / noise
    if contrast > 0:
        return float("inf")
    return 0.0

# 1. 每位患者取第 1、2 张已保存的切片，共 7 × 2 = 14 张。
evaluation_cases = []
ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
for patient_id in range(7):
    for slice_number in (1, 2):
        image_path = output_dir / f"volume-{patient_id}-slice-{slice_number}.png"
        mask_path = output_dir / f"segmentation-{patient_id}-slice-{slice_number}.png"

        with Image.open(image_path) as image_file:
            normalized_image = np.array(image_file)
        with Image.open(mask_path) as mask_file:
            mask = np.array(mask_file)
        if normalized_image.shape != mask.shape:
            raise ValueError(f"CT and mask shapes differ for volume {patient_id}, slice {slice_number}")

        # 标签 1（肝脏）和 2（肿瘤）都算作整个肝脏区域。
        liver_mask = mask > 0
        # 先扩大肝脏区域，再减去肝脏本身，得到周围的一圈组织。
        surrounding_mask = cv2.dilate(liver_mask.astype(np.uint8), ring_kernel) > 0
        surrounding_mask &= ~liver_mask & (normalized_image > 0)
        if not liver_mask.any() or not surrounding_mask.any():
            raise ValueError(f"Empty CNR region for volume {patient_id}, slice {slice_number}")

        evaluation_cases.append((normalized_image, liver_mask, surrounding_mask))

# 2. 第一张切片用来展示各滤波器的效果。
original_image = evaluation_cases[0][0]

# 3. 先计算 14 张原图的平均 CNR，作为比较基准。
original_cnr_values = []
for image, liver_mask, surrounding_mask in evaluation_cases:
    cnr = calculate_cnr(image, liver_mask, surrounding_mask)
    original_cnr_values.append(cnr)
mean_cnr_original = float(np.mean(original_cnr_values))

print(f"Evaluation set: {len(evaluation_cases)} slices (2 from each of 7 volumes)")
print(f"Mean original CNR: {mean_cnr_original:.2f}")


**Shared evaluation and display helpers.** Each filter section below only defines its filter, shows one example, and reports the 14-slice means.

In [ ]:
# 对同一组 14 张切片逐张滤波，再计算平均 PSNR 和 CNR。
def evaluate_filter(filter_function):
    psnr_values = []
    cnr_values = []
    for image, liver_mask, surrounding_mask in evaluation_cases:
        filtered_image = filter_function(image)
        psnr_values.append(calculate_psnr(image, filtered_image))
        cnr_values.append(calculate_cnr(filtered_image, liver_mask, surrounding_mask))

    mean_psnr = float(np.mean(psnr_values))
    mean_cnr = float(np.mean(cnr_values))
    return mean_psnr, mean_cnr

# 打印某一种滤波器的平均结果，方便和原图比较。
def report_filter_metrics(name, filter_function):
    psnr, cnr = evaluate_filter(filter_function)
    print(f"Mean {name} PSNR vs original (14 slices): {psnr:.2f} dB")
    print(f"Mean original CNR (14 slices): {mean_cnr_original:.2f}")
    print(f"Mean {name} CNR (14 slices): {cnr:.2f}")
    return psnr, cnr

# 把同一张样例切片的滤波前后结果并排显示。
def show_filter_example(filtered_image, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(original_image, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Original normalized CT")
    axes[0].axis("off")

    axes[1].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(title)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


#### 2.2.1 Bilateral filtering


In [ ]:
# 双边滤波：平滑图像，同时尽量保留边缘。
def apply_bilateral_filter(image, diameter=9, sigma_color=75, sigma_space=75):
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a 2D uint8 image normalized to 0-255")
    filtered_image = cv2.bilateralFilter(image, diameter, sigma_color, sigma_space)
    return filtered_image

# 在第一张样例切片上查看效果。
bilateral_image = apply_bilateral_filter(original_image)
show_filter_example(bilateral_image, "Bilateral filtered CT")


**Evaluation of bilateral filter**

In [ ]:
# 使用相同的 14 张切片计算双边滤波的平均指标。
psnr_bilateral, cnr_bilateral = report_filter_metrics("Bilateral", apply_bilateral_filter)


#### 2.2.2 Median smoothing


In [ ]:
# 中值滤波：用邻域像素的中值替换当前像素。
def apply_median_filter(image, kernel_size=5):
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a 2D uint8 image normalized to 0-255")
    # OpenCV 要求窗口大小是至少为 3 的奇数。
    if not isinstance(kernel_size, (int, np.integer)) or kernel_size < 3 or kernel_size % 2 == 0:
        raise ValueError("kernel_size must be an odd integer of at least 3")
    filtered_image = cv2.medianBlur(image, kernel_size)
    return filtered_image

median_image = apply_median_filter(original_image)
show_filter_example(median_image, "Median smoothed CT")


**Evaluation of median filter**

In [ ]:
# 使用相同的 14 张切片计算中值滤波的平均指标。
psnr_median, cnr_median = report_filter_metrics("Median", apply_median_filter)


#### 2.2.3 CLAHE


In [ ]:
# CLAHE：分块提高局部对比度，并限制对噪声的过度增强。
def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a 2D uint8 image normalized to 0-255")
    if clip_limit <= 0:
        raise ValueError("clip_limit must be positive")
    if not isinstance(tile_grid_size, (tuple, list)) or len(tile_grid_size) != 2:
        raise ValueError("tile_grid_size must contain two positive integers")
    for size in tile_grid_size:
        if not isinstance(size, (int, np.integer)) or size <= 0:
            raise ValueError("tile_grid_size must contain two positive integers")

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    filtered_image = clahe.apply(image)
    return filtered_image

clahe_image = apply_clahe(original_image)
show_filter_example(clahe_image, "CLAHE enhanced CT")


**Evaluation of CLAHE**

In [ ]:
# 使用相同的 14 张切片计算 CLAHE 的平均指标。
psnr_clahe, cnr_clahe = report_filter_metrics("CLAHE", apply_clahe)


#### 2.2.4 Global histogram equalization


In [ ]:
# 全局直方图均衡化：调整整张图的灰度分布。
def apply_histogram_equalization(image):
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a 2D uint8 image normalized to 0-255")
    filtered_image = cv2.equalizeHist(image)
    return filtered_image

he_image = apply_histogram_equalization(original_image)
show_filter_example(he_image, "Histogram equalized CT")


**Evaluation of histogram equalization (HE)**

In [ ]:
# 使用相同的 14 张切片计算全局直方图均衡化的平均指标。
psnr_he, cnr_he = report_filter_metrics("HE", apply_histogram_equalization)


### 2.3 Frequency-domain enhancement

**Both filters follow the same three steps:**
1. Compute the 2D DFT
2. Show the example spectrum
3. Filter and reconstruct the CT image.

#### 2.3.1 Butterworth low-pass filter

A Butterworth low-pass filter reduces high-frequency detail to smooth the image.

**Step 1: Compute the 2D DFT of the example CT slice.**

In [ ]:
# 计算二维离散傅里叶变换（2D DFT），并把低频移到图像中央。
# Butterworth 和同态滤波都会用到这个步骤。
def compute_2d_dft(image):
    if image.ndim != 2 or not np.issubdtype(image.dtype, np.number) or np.iscomplexobj(image):
        raise ValueError("Expected a real-valued 2D image")
    frequency_data = np.fft.fft2(image.astype(np.float32))
    centered_data = np.fft.fftshift(frequency_data)
    return centered_data

# 先查看第一张样例切片的频域数据。
blpf_frequency_data = compute_2d_dft(original_image)


**Step 2: Display the frequency spectrum.** The logarithm below only makes the spectrum easier to see.

In [ ]:
# 频谱数值跨度很大，取对数后更容易观察；这一步不改变原图。
blpf_log_spectrum = np.log1p(np.abs(blpf_frequency_data))
print("Shape:", blpf_log_spectrum.shape)
print("Range:", blpf_log_spectrum.min(), "to", blpf_log_spectrum.max())

fig, ax = plt.subplots()
spectrum_plot = ax.imshow(blpf_log_spectrum, cmap="gray")
fig.colorbar(spectrum_plot, ax=ax, label="ln(1 + |DFT coefficient|)")
ax.set_title("CT slice log-magnitude spectrum")
plt.show()


**Step 3: Reduce high frequencies, then reconstruct the CT image.**

In [ ]:
# Butterworth 低通滤波：保留较多低频，逐渐减弱高频。
def apply_butterworth_lowpass(image, cutoff=60.0, order=2):
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be positive and finite")
    if not isinstance(order, (int, np.integer)) or order < 1:
        raise ValueError("order must be a positive integer")

    # 1. 把空间域图像转换到频域。
    frequency_data = compute_2d_dft(image)

    # 2. 计算每个位置到频谱中心的距离，并建立低通滤波器。
    rows, columns = image.shape
    y, x = np.ogrid[:rows, :columns]
    distance = np.hypot(y - rows // 2, x - columns // 2)
    response = 1.0 / (1.0 + (distance / cutoff) ** (2 * order))
    filtered_frequency_data = frequency_data * response

    # 3. 逆变换回图像，再限制成 0–255 的整数。
    uncentered_data = np.fft.ifftshift(filtered_frequency_data)
    reconstructed = np.fft.ifft2(uncentered_data).real
    filtered_image = np.clip(np.rint(reconstructed), 0, 255).astype(np.uint8)
    return filtered_image

blpf_cutoff = 60.0
blpf_image = apply_butterworth_lowpass(original_image, cutoff=blpf_cutoff)
show_filter_example(blpf_image, "BLPF filtered CT")


**Butterworth mean PSNR and CNR across the 14 paired slices.**

In [ ]:
# 用小函数固定截止频率，让评估函数可以逐张调用。
def filter_blpf_for_evaluation(image):
    return apply_butterworth_lowpass(image, cutoff=blpf_cutoff)

psnr_blpf, cnr_blpf = report_filter_metrics("BLPF", filter_blpf_for_evaluation)


#### 2.3.2 Homomorphic filter



Homomorphic filtering applies a logarithm to pixel values before the 2D DFT. It slightly reduces low frequencies and boosts high frequencies. After the inverse DFT, an exponential returns to image intensities; a small adjustment keeps the average brightness of non-black tissue close to the input. The logarithm used to display the spectrum is separate from the filter's input transform.

**Step 1: Log-transform the example CT slice and compute its 2D DFT.**

In [ ]:
# 同态滤波先对像素取 log(1 + 像素值)，再计算二维 DFT。
homomorphic_input_image = np.log1p(original_image.astype(np.float32))
homomorphic_frequency_data = compute_2d_dft(homomorphic_input_image)


**Step 2: Display the frequency spectrum of the log-transformed slice.**

In [ ]:
# 这里只是把频谱可视化；滤波实际使用上一个单元的频域数据。
homomorphic_log_spectrum = np.log1p(np.abs(homomorphic_frequency_data))
print("Shape:", homomorphic_log_spectrum.shape)
print("Range:", homomorphic_log_spectrum.min(), "to", homomorphic_log_spectrum.max())

fig, ax = plt.subplots()
spectrum_plot = ax.imshow(homomorphic_log_spectrum, cmap="gray")
fig.colorbar(spectrum_plot, ax=ax, label="ln(1 + |DFT coefficient|)")
ax.set_title("CT slice log-magnitude spectrum")
plt.show()


**Step 3: Adjust low and high frequencies, then reconstruct the CT image.**

In [ ]:
# 同态滤波：轻微降低低频，同时提高高频。
def apply_homomorphic_filter(image, cutoff=60.0, gamma_low=0.9, gamma_high=1.1):
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a 2D uint8 image normalized to 0-255")
    if not np.isfinite(cutoff) or cutoff <= 0:
        raise ValueError("cutoff must be positive and finite")
    if not (0 < gamma_low < 1 < gamma_high):
        raise ValueError("Require 0 < gamma_low < 1 < gamma_high")

    # 1. 取对数并转换到频域。
    log_image = np.log1p(image.astype(np.float32))
    frequency_data = compute_2d_dft(log_image)

    # 2. 计算各频率到中心的距离，并调整低频与高频的权重。
    rows, columns = image.shape
    y, x = np.ogrid[:rows, :columns]
    distance_squared = (y - rows // 2) ** 2 + (x - columns // 2) ** 2
    response = gamma_low + (gamma_high - gamma_low) * (
        1 - np.exp(-distance_squared / cutoff**2)
    )
    filtered_frequency_data = frequency_data * response

    # 3. 逆变换后，校正非黑色组织的平均亮度。
    uncentered_data = np.fft.ifftshift(filtered_frequency_data)
    filtered_log = np.fft.ifft2(uncentered_data).real
    tissue_pixels = image > 0
    if np.any(tissue_pixels):
        original_mean = log_image[tissue_pixels].mean()
        filtered_mean = filtered_log[tissue_pixels].mean()
        filtered_log += original_mean - filtered_mean

    # 4. 用 expm1 还原取对数前的像素尺度，并保存为 8 位图像。
    reconstructed = np.expm1(filtered_log)
    filtered_image = np.clip(np.rint(reconstructed), 0, 255).astype(np.uint8)
    return filtered_image

homomorphic_cutoff = 60.0
homomorphic_image = apply_homomorphic_filter(original_image, cutoff=homomorphic_cutoff)
show_filter_example(homomorphic_image, "Homomorphic filtered CT")


**Homomorphic mean PSNR and CNR across the 14 paired slices.**

In [ ]:
# 用小函数固定截止频率，让评估函数可以逐张调用。
def filter_homomorphic_for_evaluation(image):
    return apply_homomorphic_filter(image, cutoff=homomorphic_cutoff)

psnr_homomorphic, cnr_homomorphic = report_filter_metrics(
    "Homomorphic", filter_homomorphic_for_evaluation
)


### 2.4 Final Comparison & Analysis

Each method was evaluated on **14 normalized CT images** (slices 1 and 2 from each of volumes 0–6). PSNR and CNR were calculated for each slice using its own normalized input and paired ground-truth mask, then averaged across the 14 slices. The mean CNR before filtering is **0.89**. PSNR measures similarity to the input, so a higher PSNR alone does not establish better segmentation accuracy. 

| Method | Mean PSNR (dB) | Mean CNR |
|---|---:|---:|
| Bilateral filter | 26.68 | 1.01 |
| Median smoothing | 25.36 | 0.98 |
| CLAHE | 22.85 | 0.74 |
| Histogram equalization | 21.59 | 1.01 |
| Butterworth low-pass | 25.26 | **1.02** |
| Homomorphic filter | **29.56** | 0.82 |

**Decision:** 
- Bilateral filtering show the most balanced performance across all algorithm. 
- Its mean CNR increases from 0.89 to 1.01 which is extremely close to the highest mean CNR of 1.02 from Butterworth. 
- Bilateral filtering preserves the liver boundary better than Butterworth.
- Although homomorphic filtering has the highest mean PSNR but its mean CNR is lower than bilateral filtering.

## 3. Phase 2: Classical Liver Segmentation and Morphology
---

### 3.1 Segmentation Algorithms

Compare two raw segmentation methods on the same **14 bilateral-filtered slices**:
1. Multi-Otsu: divide the image into three intensity classes.
2. Marker-controlled watershed: grow regions from fixed foreground/background markers on a Sobel gradient.

Ground-truth labels 1 (liver) and 2 (tumor) are combined into a whole-liver mask for evaluation only. Neither segmentation function receives the ground truth. Compare Dice and IoU, select a baseline, then test morphological refinement in Section 3.2.

In [ ]:
# 1. 沿用第 2 节的 14 张切片及其真实肝脏区域。
# 两种分割方法都输入同一张双边滤波后的 CT 图像。
segmentation_cases = {}
for patient_id in range(7):
    for slice_number in (1, 2):
        case_index = 2 * patient_id + slice_number - 1
        image, liver_mask, _ = evaluation_cases[case_index]
        if image.ndim != 2 or image.dtype != np.uint8:
            raise ValueError("Expected a 2D uint8 CT image")
        if image.shape != liver_mask.shape:
            raise ValueError("CT and ground-truth mask shapes must match")

        filtered_image = apply_bilateral_filter(image)
        case = (patient_id, slice_number)
        segmentation_cases[case] = (filtered_image, liver_mask)

# 2. 用患者 0 的第 1 张切片展示两种算法的过程。
example_case = (0, 1)
print(f"Prepared {len(segmentation_cases)} bilateral-filtered slices for both methods")


#### 3.1.1 Multi-Otsu Thresholding (Threshold)

In [ ]:
from skimage.filters import threshold_multiotsu

# Multi-Otsu：先分成三个灰度类别，再把最亮的一类作为肝脏候选区。
def apply_multi_otsu_threshold(image):
    # 1. 计算两个阈值，得到三个灰度类别。
    thresholds = threshold_multiotsu(image, classes=3)
    # 2. 每个像素分到 0、1 或 2 类。
    intensity_classes = np.digitize(image, bins=thresholds)
    # 3. 取类别 2 作为原始预测；这里还没有做形态学处理。
    liver_candidate = intensity_classes == 2
    return liver_candidate, thresholds


In [ ]:
# 1. 对全部 14 张切片使用同一套 Multi-Otsu 规则。
multi_otsu_masks = {}
multi_otsu_thresholds = {}
for case, (filtered_image, _) in segmentation_cases.items():
    predicted_mask, thresholds = apply_multi_otsu_threshold(filtered_image)
    multi_otsu_masks[case] = predicted_mask
    multi_otsu_thresholds[case] = thresholds

# 2. 展示一张切片的三个灰度类别和分割轮廓。
filtered_image, liver_mask = segmentation_cases[example_case]
thresholds = multi_otsu_thresholds[example_case]
intensity_classes = np.digitize(filtered_image, bins=thresholds)
print(f"Example thresholds: {thresholds}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Bilateral-filtered CT")
class_plot = axes[1].imshow(intensity_classes, cmap="viridis", vmin=0, vmax=2)
fig.colorbar(class_plot, ax=axes[1], ticks=[0, 1, 2], fraction=0.046, pad=0.04)
axes[1].set_title("Multi-Otsu intensity classes")
axes[2].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
# 绿色是真实轮廓，红色是算法预测的轮廓。
axes[2].contour(liver_mask, levels=[0.5], colors="lime", linewidths=1)
axes[2].contour(multi_otsu_masks[example_case], levels=[0.5], colors="red", linewidths=1)
axes[2].set_title("Ground truth (green) + prediction (red)")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


#### 3.1.2 Marker-Controlled Watershed (Region-Based)

**Step 1: Create markers from the CT image.** Marker 0 is unknown, 1 is non-liver, and 2 is a liver seed. Use the same fixed rules on all slices:
- In a rectangular core on the image's left side, pixels with grayscale values **70-190** become liver seeds.
- Tissue pixels outside a larger ellipse become non-liver seeds. Other pixels remain unknown.

The rectangle, ellipse, and intensity limits are retained from the previous implementation; connected-component ranking and its fallback have been removed. They are location/intensity assumptions for this dataset orientation, not standard watershed constants. The rectangle uses rows 38%-72% and columns 14%-42%; the ellipse is centered at (31% width, 55% height), with radii 40% width and 44% height. Values above 5 define a rough tissue processing mask.

**Step 2: Compute the Sobel gradient.** Strong intensity changes form high barriers.

**Step 3: Run watershed and keep label 2.** These raw masks have no erosion, dilation, opening, or closing. Seed placement can include other tissue, so the overlay and Dice/IoU must be checked.

In [ ]:
from skimage.filters import sobel
from skimage.segmentation import watershed

# 根据 CT 的位置和亮度，建立分水岭算法的起点（markers）。
def create_watershed_markers(image):
    rows, columns = image.shape

    # 1. 图像左侧矩形中的 70–190 灰度像素，作为肝脏种子。
    seed_region = np.zeros(image.shape, dtype=bool)
    seed_region[
        #int(0.38 * rows):int(0.72 * rows),
        #int(0.14 * columns):int(0.42 * columns),
        int(0.43 * rows):int(0.65 * rows),
        int(0.14 * columns):int(0.42 * columns),
    ] = True
    #liver_seed = seed_region & (image >= 70) & (image <= 150) 
    liver_seed = seed_region & (image >= 70) & (image <= 175) 

    # 2. 宽椭圆之外的身体组织，作为非肝脏种子。
    y, x = np.ogrid[:rows, :columns]
    search_region = (
        #((x - 0.31 * columns) / (0.40 * columns)) ** 2
        #+ ((y - 0.55 * rows) / (0.44 * rows)) ** 2 <= 1
        ((x - 0.31 * columns) / (0.35 * columns)) ** 2
        + ((y - 0.55 * rows) / (0.44 * rows)) ** 2 <= 1
    )
    body = image > 5
    background_seed = body & ~search_region
    if not liver_seed.any() or not background_seed.any():
        raise ValueError("Watershed needs both liver and non-liver seeds")

    # 3. 0=未知，1=非肝脏，2=肝脏。
    markers = np.zeros(image.shape, dtype=np.int32)
    markers[background_seed] = 1
    markers[liver_seed] = 2
    return markers, body

# 根据种子和图像边缘，得到原始的肝脏预测区域。
def apply_marker_controlled_watershed(image):
    markers, body = create_watershed_markers(image)
    gradient = sobel(image.astype(np.float32) / 255.0)
    labels = watershed(gradient, markers, mask=body, connectivity=2)
    liver_candidate = labels == 2
    return liver_candidate, markers


In [ ]:
# 1. 对全部 14 张切片使用同一套分水岭种子规则。
watershed_masks = {}
watershed_markers = {}
for case, (filtered_image, _) in segmentation_cases.items():
    predicted_mask, markers = apply_marker_controlled_watershed(filtered_image)
    watershed_masks[case] = predicted_mask
    watershed_markers[case] = markers

# 2. 展示一张切片的种子、Sobel 梯度和分割轮廓。
filtered_image, liver_mask = segmentation_cases[example_case]
markers = watershed_markers[example_case]
gradient = sobel(filtered_image.astype(np.float32) / 255.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
marker_plot = axes[0].imshow(
    np.ma.masked_where(markers == 0, markers),
    cmap="coolwarm", vmin=1, vmax=2, alpha=0.6,
)
fig.colorbar(marker_plot, ax=axes[0], ticks=[1, 2], fraction=0.046, pad=0.04)
axes[0].set_title("Markers: 1=non-liver, 2=liver")
axes[1].imshow(gradient, cmap="gray")
axes[1].set_title("Sobel gradient")
axes[2].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
# 绿色是真实轮廓，红色是算法预测的轮廓。
axes[2].contour(liver_mask, levels=[0.5], colors="lime", linewidths=1)
axes[2].contour(watershed_masks[example_case], levels=[0.5], colors="red", linewidths=1)
axes[2].set_title("Ground truth (green) + prediction (red)")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


#### 3.1.3 Compare and Select the Raw Segmentation

Calculate Dice and IoU separately for each slice, then average the 14 scores. Use the higher mean Dice to choose the baseline for Section 3.2. This compares the two configured methods: watershed includes a fixed location prior, while Multi-Otsu uses intensity alone. These slices are used for method selection, not an independent test of generalization.

In [ ]:
# Dice 和 IoU 用来比较预测区域与真实肝脏区域的重叠程度。
def calculate_dsc_iou(predicted_mask, ground_truth_mask):
    if predicted_mask.shape != ground_truth_mask.shape:
        raise ValueError("Predicted and ground-truth masks must have the same shape")

    predicted = predicted_mask.astype(bool)
    ground_truth = ground_truth_mask.astype(bool)
    intersection = np.count_nonzero(predicted & ground_truth)
    predicted_pixels = np.count_nonzero(predicted)
    ground_truth_pixels = np.count_nonzero(ground_truth)
    total_pixels = predicted_pixels + ground_truth_pixels
    union = np.count_nonzero(predicted | ground_truth)

    # 两张都是空白区域时，视为完全一致。
    if union == 0:
        return 1.0, 1.0
    dice = 2 * intersection / total_pixels
    iou = intersection / union
    return dice, iou

# 逐张算分数，然后取 14 张切片的平均值。
def evaluate_segmentation(predicted_masks):
    scores = {}
    dice_values = []
    iou_values = []
    for case, (_, ground_truth_mask) in segmentation_cases.items():
        dice, iou = calculate_dsc_iou(predicted_masks[case], ground_truth_mask)
        scores[case] = (dice, iou)
        dice_values.append(dice)
        iou_values.append(iou)

    mean_dsc = float(np.mean(dice_values))
    mean_iou = float(np.mean(iou_values))
    return scores, mean_dsc, mean_iou


In [ ]:
# 1. 用相同的 14 张切片比较两种原始分割结果。
multi_otsu_scores, mean_multi_otsu_dsc, mean_multi_otsu_iou = evaluate_segmentation(multi_otsu_masks)
watershed_scores, mean_watershed_dsc, mean_watershed_iou = evaluate_segmentation(watershed_masks)

# 2. 先打印每张切片的 Dice 和 IoU，再打印平均值。
print(f"{'Case':<18} {'Otsu Dice':>10} {'Otsu IoU':>10} {'WS Dice':>10} {'WS IoU':>10}")
for patient_id, slice_number in segmentation_cases:
    case = (patient_id, slice_number)
    otsu_dsc, otsu_iou = multi_otsu_scores[case]
    ws_dsc, ws_iou = watershed_scores[case]
    label = f"Volume {patient_id}, slice {slice_number}"
    print(f"{label:<18} {otsu_dsc:>10.4f} {otsu_iou:>10.4f} {ws_dsc:>10.4f} {ws_iou:>10.4f}")
print(f"{'Mean (14 slices)':<18} {mean_multi_otsu_dsc:>10.4f} {mean_multi_otsu_iou:>10.4f} {mean_watershed_dsc:>10.4f} {mean_watershed_iou:>10.4f}")

# 3. 用平均 Dice 较高的方法，作为第 3.2 节形态学处理的起点。
if mean_watershed_dsc > mean_multi_otsu_dsc:
    selected_method = "Watershed"
    selected_masks = watershed_masks
else:
    selected_method = "Multi-Otsu"
    selected_masks = multi_otsu_masks

# 复制一份原始结果，方便之后比较处理前后，且不会改动原始字典。
selected_raw_masks = {}
for case, mask in selected_masks.items():
    selected_raw_masks[case] = mask.copy()
print(f"\nSelected raw segmentation for morphology: {selected_method}")


### 3.2 Refine masks with morphology

Use `selected_raw_masks` as the unchanged baseline. Apply morphological operations to a separate set of masks, then use `evaluate_segmentation` to compare Dice and IoU before and after refinement on the same 14 slices.

Choose the operation and structuring element from the observed error: opening can remove small fragments, and closing can address small holes or gaps. Also examine erosion and dilation as required by the assignment. Record improvements and regressions; an increase is not guaranteed. Broad leakage into other tissues may remain even after these local shape corrections.

**To implement:** Create refined masks, show overlays, and report the before/after Dice and IoU. Explain the chosen structuring element and which visible errors improved or worsened.

## 4. Phase 3: Features and classical classification (planned)
---

### 4.1 Extract ROI features

Extract shape descriptors, multi-angle GLCM texture features, and first-order CT intensity statistics from the segmented regions. Define a clear target label for each ROI before training.

### 4.2 Train and validate a classical classifier

Normalize feature vectors within a training pipeline. Split by patient before using a 70/30 train/test split, then use 5-fold cross-validation on the training patients to avoid slices from one patient appearing in both sets. Use a classical model such as an SVM or decision tree.

### 4.3 Evaluate diagnosis

**To implement:** Show the confusion matrix and report accuracy, sensitivity (recall), specificity, and F1 score on held-out patients.